<a href="https://colab.research.google.com/github/jesusessu/MDD_LAB07/blob/develop/MDD_LAB07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Minería de Datos - Semana 7: Regresión Logística y SVM**

Alumno:

Esplana Sulla Jesús Zósimo

In [17]:
# --- Librerías necesarias ---
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn import metrics
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [19]:
# --- Carga y limpieza de datos ---
# URL del dataset
URL_DATA = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

In [20]:
# Definimos columnas
COLS = ['id', 'clump_thickness', 'uniformity_cell_size', 'uniformity_cell_shape',
        'marginal_adhesion', 'single_epithelial_cell_size', 'bare_nuclei',
        'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']

In [21]:
# Funciones de carga y preprocesamiento
def cargar_y_preparar(url):
    df = pd.read_csv(url, names=COLS)
    df.replace('?', np.nan, inplace=True)
    df.dropna(inplace=True)
    df['bare_nuclei'] = df['bare_nuclei'].astype(int)
    df['class'] = df['class'].apply(lambda x: 1 if x == 4 else 0)
    return df.drop(columns=['id'])

In [23]:
# Cargar los datos
data = cargar_y_preparar(URL_DATA)

In [24]:
# --- Calculo del IV ---
def calcular_iv(df, target):
    def iv_variable(df, feature, target):
        tabla = df.groupby(feature)[target].value_counts().unstack(fill_value=0)
        if 0 not in tabla.columns:
            tabla[0] = 0
        if 1 not in tabla.columns:
            tabla[1] = 1
        tabla = tabla[[0,1]]

        total_event = tabla[1].sum()
        total_nonevent = tabla[0].sum()

        tabla['dist_event'] = tabla[1] / total_event
        tabla['dist_nonevent'] = tabla[0] / total_nonevent

        tabla['dist_event'].replace(0, 1e-6, inplace=True)
        tabla['dist_nonevent'].replace(0, 1e-6, inplace=True)

        tabla['woe'] = np.log(tabla['dist_event'] / tabla['dist_nonevent'])
        tabla['iv'] = (tabla['dist_event'] - tabla['dist_nonevent']) * tabla['woe']

        return tabla['iv'].sum()

    iv_resultados = {col: iv_variable(df, col, target) for col in df.columns if col != target}
    return pd.Series(iv_resultados).sort_values(ascending=False)


In [16]:
# Modelo SVM
normalizador = StandardScaler()
X_train_norm = normalizador.fit_transform(X_train)
X_test_norm = normalizador.transform(X_test)

clasificador_svm = SVC()
clasificador_svm.fit(X_train_norm, y_train)
predicciones_svm = clasificador_svm.predict(X_test_norm)

print("\n--- Métricas del Modelo SVM ---")
print("Precisión:", metrics.accuracy_score(y_test, predicciones_svm))
print(metrics.classification_report(y_test, predicciones_svm))

print("\nComparativa de Modelos:")
print(f"Regresión Logística - Precisión: {metrics.accuracy_score(y_test, predicciones_lr):.4f}")
print(f"SVM - Precisión: {metrics.accuracy_score(y_test, predicciones_svm):.4f}")


--- Métricas del Modelo SVM ---
Precisión: 0.9766081871345029
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       108
           1       0.98      0.95      0.97        63

    accuracy                           0.98       171
   macro avg       0.98      0.97      0.97       171
weighted avg       0.98      0.98      0.98       171


Comparativa de Modelos:
Regresión Logística - Precisión: 0.9766
SVM - Precisión: 0.9766


In [25]:
# IV de las variables
iv_scores = calcular_iv(data, 'class')
print("Information Value de las variables:\n", iv_scores)

Information Value de las variables:
 uniformity_cell_size           10.388630
uniformity_cell_shape           8.601120
clump_thickness                 6.532306
bland_chromatin                 6.286301
normal_nucleoli                 6.085052
bare_nuclei                     5.463221
marginal_adhesion               4.880258
single_epithelial_cell_size     4.235413
mitoses                         2.331167
dtype: float64


In [26]:
# Seleccionar variables fuertes
variables_utiles = iv_scores[iv_scores >= 0.02].index.tolist()

In [27]:
# --- Dividir datos en entrenamiento y prueba ---
X = data[variables_utiles]
y = data['class']

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [30]:
# --- Modelo de Regresión Logística ---
X_train_const = sm.add_constant(X_train)
modelo_logit = sm.Logit(y_train, X_train_const).fit()
print(modelo_logit.summary())

Optimization terminated successfully.
         Current function value: 0.073760
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  class   No. Observations:                  512
Model:                          Logit   Df Residuals:                      502
Method:                           MLE   Df Model:                            9
Date:                Sun, 04 May 2025   Pseudo R-squ.:                  0.8842
Time:                        03:20:45   Log-Likelihood:                -37.765
converged:                       True   LL-Null:                       -326.13
Covariance Type:            nonrobust   LLR p-value:                2.070e-118
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                         -10.0640      1.408     -7.150      0.000   

In [31]:
# Entrenamiento y predicción
lr = LogisticRegression(max_iter=300)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [32]:
# Métricas Regresión Logística
print("\n--- Métricas Regresión Logística ---")
print("Precisión:", metrics.accuracy_score(y_test, y_pred_lr))
print(metrics.classification_report(y_test, y_pred_lr))


--- Métricas Regresión Logística ---
Precisión: 0.9532163742690059
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       103
           1       0.98      0.90      0.94        68

    accuracy                           0.95       171
   macro avg       0.96      0.94      0.95       171
weighted avg       0.95      0.95      0.95       171



In [34]:
# --- Modelo SVM ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

svm_clf = SVC()
svm_clf.fit(X_train_scaled, y_train)
y_pred_svm = svm_clf.predict(X_test_scaled)

In [35]:
# Métricas SVM
print("\n--- Métricas SVM ---")
print("Precisión:", metrics.accuracy_score(y_test, y_pred_svm))
print(metrics.classification_report(y_test, y_pred_svm))


--- Métricas SVM ---
Precisión: 0.9590643274853801
              precision    recall  f1-score   support

           0       0.95      0.98      0.97       103
           1       0.97      0.93      0.95        68

    accuracy                           0.96       171
   macro avg       0.96      0.95      0.96       171
weighted avg       0.96      0.96      0.96       171



In [36]:
# --- Comparativa ---
print("\nComparativa de Precisión:")
print(f"Regresión Logística: {metrics.accuracy_score(y_test, y_pred_lr):.4f}")
print(f"SVM: {metrics.accuracy_score(y_test, y_pred_svm):.4f}")


Comparativa de Precisión:
Regresión Logística: 0.9532
SVM: 0.9591
